In [76]:
import sympy as sp
from sympy import Matrix, factorial, expand, collect, simplify, latex
from IPython.display import display, Math
# from sympy.printing.latex import LatexPrinter

# LatexPrinter.set_global_settings()
sp.init_printing(fontsize='44pt')

In [77]:
# Main symbols
dx = sp.Symbol(r'\Delta x', positive=True, real=True)
dt = sp.Symbol(r'\Delta t', positive=True, real=True)
a = sp.Symbol('a', positive=True, real=True)
sigma = sp.Symbol(r'\sigma', real=True)

# Derivatives at point (i, n)
u = sp.Symbol('u')
u_x = sp.Symbol(r"u'_x")
u_xx = sp.Symbol(r"u''_{xx}")
u_xxx = sp.Symbol(r"u'''_{xxx}")
u_xxxx = sp.Symbol(r"u^{IV}_{xxxx}")
u_5x = sp.Symbol(r"u^{V}_{xxxxx}")
u_6x = sp.Symbol(r"u^{VI}_{xxxxxx}")

derivs = [u, u_x, u_xx, u_xxx, u_xxxx, u_5x, u_6x]

In [78]:
def taylor_spatial(offset, order=6):
    """ Taylor expansion of u(x + offset*dx, t) around (x, t) """
    result = sp.Integer(0)
    for k in range(min(order + 1, len(derivs))):
        result += derivs[k] * (offset * dx)**k / factorial(k)
    return result

u_im2 = taylor_spatial(-2)
u_im1 = taylor_spatial(-1)
u_i = u
u_ip1 = taylor_spatial(1)
u_ip2 = taylor_spatial(2)

print("Taylor expansions:")
for name, expr in [("u_{i-2}", u_im2), ("u_{i-1}", u_im1), 
                   ("u_{i+1}", u_ip1), ("u_{i+2}", u_ip2)]:
    display(Math(f"{name} \\approx {latex(expr, order='rev-lex')}"))

Taylor expansions:


<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [79]:
term_avg = sigma * u_i + (1 - sigma) * (u_ip1 + u_im1) / 2
term_ux = -a * dt * (u_ip1 - u_im1) / (2 * dx)
term_uxx = (a * dt)**2 / 2 * (u_ip1 - 2*u_i + u_im1) / dx**2

u_next = term_avg + term_ux + term_uxx

u_exact = (u
           - a*dt*u_x
           + a**2*dt**2/2*u_xx
           - a**3*dt**3/6*u_xxx
           + a**4*dt**4/24*u_xxxx
           - a**5*dt**5/120*u_5x
           + a**6*dt**6/720*u_6x)

display(Math(f"u_i^{{n+1}} = {latex(collect(expand(u_next), derivs), order='lex')}"))

<IPython.core.display.Math object>

In [80]:
error = expand(u_next - u_exact)


error_coeffs = {}

table = \
"\\begin{array} {r r r} \\\\"\
" & \\text{Coefficient} & \\text{Error coefficient} \\\\"\
"\\hline \\\\"

for k, deriv in enumerate(derivs):
    def get_coeff(expr, deriv):
        return sp.together(simplify(expand(expr).coeff(deriv)))

    error_coeff = get_coeff(error, deriv)

    coeff = get_coeff(u_next, deriv)
    error_coeffs[k] = error_coeff
    table += f"\dfrac{{\partial^{k} u}}{{\partial x^{k}}} : &\\quad{latex(coeff)} &\\quad  {latex(error_coeff)} & \\\\[1em]"

table += "\\end{array}"
display(Math(table))

<IPython.core.display.Math object>

In [81]:
error_coeffs_normalized = []

for k in range(5):
    error_coeffs_normalized.append(simplify(error_coeffs[k] * factorial(k) * 6 / (a**3 * dt**3 * dx**k)))

error_coeffs_normalized = Matrix(error_coeffs_normalized)
display(Math(f"6 \\cdot {latex(error_coeffs_normalized / 6)}"))

<IPython.core.display.Math object>

In [82]:
taylor_coeffs_normalized = Matrix([
    [(-2)**k, (-1)**k, 0**k if k > 0 else 1, 1**k, 2**k]
    for k in range(5)
])

display(Math(f"{latex(taylor_coeffs_normalized)} x = 6 \\cdot {latex(error_coeffs_normalized / 6)},\\\\[2em]"))

solution = taylor_coeffs_normalized.solve(error_coeffs_normalized)

display(Math(f"x = {latex(1/ (4 * dt**3 * dx**4 * a**3))} \\cdot {latex(solution * 4 * dt**3 * dx**4 * a**3)}"))

<IPython.core.display.Math object>

<IPython.core.display.Math object>

In [ ]:
A_coef, B_coef, C_coef, D_coef, E_coef = solution

correction_term = -a**3 * dt**3 / 6 * (
    A_coef * u_im2 + 
    B_coef * u_im1 + 
    C_coef * u_i + 
    D_coef * u_ip1 + 
    E_coef * u_ip2
)

# Full corrected scheme
u_corrected = u_next + correction_term
u_corrected = expand(u_corrected)

error_corrected = expand(u_corrected - u_exact)

for d in enumerate(derivs):
    coeff = simplify(get_coeff(error_corrected, deriv))
    if coeff != 0:
        display(Math(f"\\text{{Coefficient of highest-order error term }} {latex(deriv)}:\\\\[1em] \\quad {latex(coeff)}"))
        break

<IPython.core.display.Math object>